# 05 · Kaggle real-data profiling & evaluation — dataset by dataset, read in place from `/kaggle/input`

> **REAL PUBLIC DATA workflow + one SYNTHETIC third-party dataset.** Nothing is downloaded to your laptop, copied into git, or committed: datasets stay in Kaggle's input storage (mounted read-only) and are processed *in place*; outputs go only to `/kaggle/working`. The output bundle contains **aggregate reports only** (no records, no images, no videos, no annotations, no CSV rows). Every licence on the source cards is **UNVERIFIED**, so no result here may be presented as a publishable real-world claim. Real and synthetic results are never pooled.

**First run = PROFILE / AUDIT.** It establishes *what is actually inside each dataset, and can the CivicConnect pipeline consume it correctly?* No model or provider is called (`RUN_PROVIDER_EVAL = False`), no GPU/TPU and no API credentials are needed, and **nothing is guessed**: annotation format, class-name file, ambiguous CSVs, group rules and holdouts are never chosen silently. When something must be decided, the run stops *for that dataset only* and prints the exact `IMAGE[...]`/`ROLE_MAP`/`FILES` setting to supply; the other datasets still run.

**Exact sources this notebook was written for**

| id | source (attach exactly this) | evidence class |
|---|---|---|
| `bmc_mumbai` | [Kaggle competition data](https://www.kaggle.com/competitions/mumbai-nagar-seva-bmc-civic-complaint-resolution-2018-2024/data) · [rules](https://www.kaggle.com/competitions/mumbai-nagar-seva-bmc-civic-complaint-resolution-2018-2024/rules) — files `bmc_train.csv`, `bmc_data_dictionary.csv` (`bmc_test.csv` is never opened) | **third-party SYNTHETIC** (the rules state it is synthetically generated) — never real-world |
| `mumbai_nashik_road_surface` | [Mendeley tj2m7zz4rg **v2**](https://data.mendeley.com/datasets/tj2m7zz4rg/2) — 8,484 images + 13 videos, uploaded by you as a private Kaggle Dataset | real; CC BY 4.0 reported (unverified here) |
| `rdd2022` | [Kaggle `aliabdelmenam/rdd-2022`](https://www.kaggle.com/datasets/aliabdelmenam/rdd-2022) — a **third-party copy**; its layout is *not* assumed to match the official distribution | real; licence unresolved (BY vs BY-SA), copy's own licence unverified |
| `rdd2020` | [Mendeley 5ty2wb6gvg v1](https://data.mendeley.com/datasets/5ty2wb6gvg/1) (India subset where applicable), uploaded by you as a Kaggle Dataset | real; CC BY-NC 3.0 reported (unverified) |
| `bharatpothole` | [Kaggle `surbhisaswatimohanty/bharatpothole`](https://www.kaggle.com/datasets/surbhisaswatimohanty/bharatpothole) | real; **licence pending** (publishable evaluation disabled) |

**Settings:** CPU, Internet *Off* is fine once the repo code is available (see the bootstrap below), no GPU/TPU. Image datasets are two-pass: run once to **profile**, read the report, then fill `IMAGE[...]` and re-run.

## 0 · Repository bootstrap — which code is this notebook executing?
Preference order: **(1)** `CIVIC_REPO` (explicit path) → **(2)** the repo this notebook already sits in → **(3)** an attached Kaggle Dataset containing the repo's `ai/` folder → **(4)** a `git` checkout of `CIVIC_GIT_URL` at **`CIVIC_GIT_REF`** (a branch, tag or commit SHA; needs Internet *On*). The default ref is the implementation branch `claude/epic-fermat-3qlw5c` — **never an implicit `main`**, which does not contain this notebook until the PR is merged. A checkout (or copy) that lacks Notebook 05 / the data-source framework is rejected as stale. The resolved repo, ref and commit are printed and recorded in `SUMMARY.json`.

In [ ]:
# --- 0. Bootstrap: locate/checkout the repo, verify it, check dependencies (CPU only, no credentials) --------------
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
GIT_URL = os.environ.get("CIVIC_GIT_URL", "https://github.com/tanujb03/CivicConnect_v2")
GIT_REF = os.environ.get("CIVIC_GIT_REF", "claude/epic-fermat-3qlw5c")           # branch | tag | commit SHA — NOT an implicit main
REQUIRED_FILES = ["ai/inference/schemas.py", "ai/training/src/data_sources/kaggle_run.py", "ai/training/notebooks/05_kaggle_real_data_profile_evaluate.ipynb"]
WORK = Path(os.environ.get("CIVIC_WORKDIR") or ("/kaggle/working" if ON_KAGGLE else tempfile.mkdtemp(prefix="civic_work_")))
WORK.mkdir(parents=True, exist_ok=True)

def git(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, check=True, capture_output=True, text=True).stdout.strip()

def verify(repo: Path) -> Path:
    missing = [f for f in REQUIRED_FILES if not (repo / f).is_file()]
    if missing:
        raise SystemExit(f"{repo} is a STALE or wrong checkout (missing {missing}). Use CIVIC_GIT_REF=claude/epic-fermat-3qlw5c (or the commit SHA you were given).")
    return repo

def find_repo():
    if os.environ.get("CIVIC_REPO"):
        return verify(Path(os.environ["CIVIC_REPO"])), "CIVIC_REPO (explicit path)"
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:                                                  # already inside the repo: prefer the local code
        if (p / "ai/inference/schemas.py").is_file():
            return verify(p), "local checkout (this notebook's own repo)"
    if ON_KAGGLE:
        for hit in sorted(Path("/kaggle/input").glob("**/ai/inference/schemas.py"))[:1]:
            dst = WORK / "repo"
            if not dst.exists():
                shutil.copytree(hit.parents[2], dst)                                 # code only (small); never a dataset
            return verify(dst), f"attached Kaggle Dataset {hit.parents[2]}"
    dst = Path(os.environ.get("CIVIC_CLONE_DIR") or (WORK / "repo"))
    try:                                                                             # needs Internet On; refs may be branch, tag or SHA
        dst.mkdir(parents=True, exist_ok=True)
        if not (dst / ".git").exists():
            git("init", "-q", cwd=dst)
        git("fetch", "-q", "--depth", "1", GIT_URL, GIT_REF, cwd=dst)
        git("checkout", "-q", "--force", "FETCH_HEAD", cwd=dst)
    except (subprocess.CalledProcessError, OSError) as e:
        raise SystemExit(f"Cannot get the repo code ({GIT_URL} @ {GIT_REF}): {getattr(e, 'stderr', e)}\nOptions: turn Internet On, or add a Kaggle Dataset containing the repo's `ai/` folder, or set CIVIC_REPO.") from e
    return verify(dst), f"git checkout {GIT_URL} @ {GIT_REF}"

REPO, SOURCE = find_repo()
sys.path.insert(0, str(REPO))
try:
    COMMIT, BRANCH = git("rev-parse", "HEAD", cwd=REPO), git("rev-parse", "--abbrev-ref", "HEAD", cwd=REPO)
except (subprocess.CalledProcessError, OSError):
    COMMIT, BRANCH = "unknown (no .git: copied from a Kaggle Dataset)", "unknown"
REPO_INFO = {"repo_path": str(REPO), "source": SOURCE, "requested_url": GIT_URL, "requested_ref": GIT_REF, "commit": COMMIT, "head": BRANCH}
print("EXECUTING REPO :", json.dumps(REPO_INFO, indent=1))

# --- dependencies: required ones must import (clear instruction otherwise); Pillow is optional and its absence is reported, not hidden
if sys.version_info < (3, 11):
    raise SystemExit(f"Python >= 3.11 required, found {sys.version.split()[0]}")
missing = [m for m in ("numpy", "pydantic", "httpx") if importlib.util.find_spec(m) is None]
if missing:
    print("installing missing required packages:", missing)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f"Missing required packages {missing} and pip could not install them (Internet Off?). Run:  pip install {' '.join(missing)}")
pydantic = importlib.import_module("pydantic")
if int(pydantic.VERSION.split(".")[0]) < 2:
    raise SystemExit(f"pydantic >= 2 required (found {pydantic.VERSION}). Run:  pip install -U 'pydantic>=2'")
HAVE_PILLOW = importlib.util.find_spec("PIL") is not None
print(f"python {sys.version.split()[0]} | pydantic {pydantic.VERSION} | Pillow {'available' if HAVE_PILLOW else 'MISSING: the near-duplicate audit will be reported as NOT RUN (pip install pillow to enable)'}")
print(f"work: {WORK} | kaggle: {ON_KAGGLE} | no GPU, no TPU, no API credentials are used")

## 1 · Settings (edit these)
* `ACCEPT_TERMS`: dataset ids whose terms **you** have read on the dataset's own page. Profiling never needs this; *preparing* records does.
* `ACK_UNVERIFIED_LICENSE`: required while a card's licence is `UNVERIFIED` (always, today). Results then must not be published.
* `PATHS` / `FILES`: only if discovery by name cannot find your mount or the BMC training file.
* `ROLE_MAP` (BMC): `{column: role}` after reading `bmc_data_dictionary.csv`; `RESOLUTION_UNIT`: `'days'`/`'hours'` if a duration column exists (a post-resolution *target*, never an input to intake/triage).
* `EXTRACT[id] = True` / `ARCHIVE[id]`: only if the profile shows the dataset is an **archive** (zip/tar, even without an extension) that Kaggle did not unpack. The first run lists it without extracting; `EXTRACT` then unpacks ONLY images + annotation/class files into `/kaggle/working/extracted/<id>` (an explicit copy inside Kaggle's working disk, never bundled, refused if it would not fit; videos only with `EXTRACT_VIDEOS = True`). `ARCHIVE[id]` picks one when a mount holds several.
* `IMAGE[id]` (**fill after the profile pass**): `format` (`voc`/`yolo`/`coco`/`folder`), `class_names_file` (YOLO), `countries` (e.g. `["India"]` for the India subset), `group_by` (`dir`/`regex`/`block`) + `group_regex`, `holdout_fraction`, `holdout_seed`, or `split_from_path` to use the source's own split folders (still audited for leakage). A holdout is only built from an explicit rule, so frames of one video can never straddle train and test.

In [ ]:
DATASETS = ["bmc_mumbai", "mumbai_nashik_road_surface", "rdd2022", "rdd2020", "bharatpothole"]   # run any subset
ACCEPT_TERMS = []                      # e.g. ["bmc_mumbai", "rdd2022"] — ONLY after reading each dataset's terms
ACK_UNVERIFIED_LICENSE = False         # set True to prepare records locally while licences are unverified (no publishing)
INPUT_ROOT = None                      # default /kaggle/input (or $CIVIC_KAGGLE_INPUT)
PATHS, FILES = {}, {}                  # e.g. PATHS = {"rdd2020": "/kaggle/input/my-rdd2020-india"}
EXTRACT, ARCHIVE, EXTRACT_VIDEOS = {}, {}, False   # only if the profile says the data is an archive: EXTRACT = {"rdd2022": True}
ROLE_MAP, RESOLUTION_UNIT = {}, None  # BMC only
IMAGE = {                              # fill AFTER the profile pass — examples only; nothing is assumed
    # "rdd2022": {"format": "yolo", "class_names_file": "data.yaml", "countries": ["India"], "group_by": "block", "block_size": 100, "holdout_fraction": 0.3},
    # "bharatpothole": {"format": "yolo", "class_names_file": "classes.txt", "group_by": "regex", "group_regex": r"^(.+?)_\d+$", "holdout_fraction": 0.3},
    # "mumbai_nashik_road_surface": {"format": "folder", "group_by": "dir", "holdout_fraction": 0.3},
}
RUN_PROVIDER_EVAL = False              # no model/provider call on the first run

# CI hook ONLY: run on an INVENTED /kaggle/input-shaped tree built from the repo's test helpers (NOT real data)
if os.environ.get("CIVIC_KAGGLE_FIXTURE") == "1":
    from ai.training.tests.conftest_real import make_fake_kaggle_input
    INPUT_ROOT = make_fake_kaggle_input(WORK / "fake_kaggle_input", rdd2020=True, bharat=True, nashik=True)
    ACCEPT_TERMS, ACK_UNVERIFIED_LICENSE, RESOLUTION_UNIT = list(DATASETS), True, None
    voc = {"format": "voc", "countries": ["India"], "group_by": "block", "block_size": 5, "holdout_fraction": 0.5, "holdout_seed": 1}
    IMAGE = {"rdd2022": voc, "rdd2020": voc,
             "bharatpothole": {"format": "yolo", "class_names_file": "classes.txt", "group_by": "regex", "group_regex": r"^(vid\d+)_", "holdout_fraction": 0.5, "holdout_seed": 2},
             "mumbai_nashik_road_surface": {"format": "folder", "group_by": "dir", "holdout_fraction": 0.5, "holdout_seed": 4}}

## 2 · Discovery — what is attached, and exactly what is missing
Read-only: nothing is copied, nothing is written.

In [ ]:
from ai.training.src.data_sources import kaggle_run as kr

CFG = kr.RunConfig(work_dir=WORK, input_root=INPUT_ROOT, accept_terms=ACCEPT_TERMS, ack_unverified_license=ACK_UNVERIFIED_LICENSE, paths=PATHS, files=FILES,
                   role_map=ROLE_MAP, resolution_unit=RESOLUTION_UNIT, image=IMAGE, run_provider_eval=RUN_PROVIDER_EVAL, meta=REPO_INFO,
                   extract=EXTRACT, archive=ARCHIVE, extract_include_videos=EXTRACT_VIDEOS)
for row in kr.plan(CFG, tuple(DATASETS)):
    print(("FOUND    " if row["found"] else "MISSING  ") + f"{row['dataset_id']:28s} priority={row['priority']:9s} licence={row['licence_status']}  {row['path'] or ''}")
    if not row["found"]:
        print("          ", row["message"])
    if row.get("ambiguity"):
        print("           AMBIGUOUS:", row["ambiguity"])

## 3 · Run, dataset by dataset
Per dataset: **card/provenance → locate → profile → (BMC) data dictionary, column-role validation, leakage audit, synthetic-track evaluation · (images) explicit format & class names, group-safe holdout, annotation / duplicate / near-duplicate / group audits, `real_holdout` readiness**. A dataset that is absent, needs a setting, or needs your terms acceptance returns a precise status (`NOT_ATTACHED`, `NEEDS_CONFIG`, `NEEDS_TERMS`, `BLOCKED`) and the others still run.

In [ ]:
SUMMARY = kr.run_all(CFG, tuple(DATASETS))
print(f"{'dataset':28s} {'status':14s} {'evidence':22s} {'licence':11s} real_holdout?")
for ds, v in SUMMARY["datasets"].items():
    print(f"{ds:28s} {v['status']:14s} {v['evidence_class']:22s} {v['licence_status']:11s} {v['can_run_real_holdout']}")
print("\n" + SUMMARY["reminder"])

## 4 · Read the reports — every stage in full (copy this output when asking what to configure next)

In [ ]:
for ds in DATASETS:
    rep = json.loads((CFG.out / ds / "report.json").read_text(encoding="utf-8"))
    print("=" * 100, f"\n{ds}: {rep['status']}  (evidence: {rep['evidence_class']})")
    for st in rep["stages"]:
        print(f"--- {st['stage']} [{st['status']}]")
        print((json.dumps(st["detail"], indent=1, ensure_ascii=False, default=str) if not isinstance(st["detail"], str) else st["detail"])[:2500])
    for n in rep["next_steps"]:
        print("NEXT >>", n)

## 5 · Bundle AGGREGATES only (allowlisted reports, manifests, profiles). Prepared records, CSVs, images, videos and annotations are excluded by construction.

In [ ]:
import hashlib
import zipfile

BUNDLE = kr.bundle(CFG)
names = zipfile.ZipFile(BUNDLE).namelist()
assert not any(n.endswith(("records.jsonl", "pairs.jsonl", "raw.jsonl", ".jpg", ".png", ".csv", ".mp4", ".xml", ".txt")) for n in names)
print("bundle (aggregates only):", BUNDLE, "|", BUNDLE.stat().st_size, "bytes | sha256", hashlib.sha256(BUNDLE.read_bytes()).hexdigest()[:16], "|", len(names), "files")
print("REMINDER: BMC is synthetic third-party data; every licence is unverified; do not commit or redistribute data; keep licence status in every report.")